In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_I.xpt"
csv_file = DATA_DIR / "HOQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 HOQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOD050",
    "HOQ065"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HOQ_I columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9971:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,971 rows x 3 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached HOQ_I:
#
#     genuine decimal variables = NONE
#     string variables          = NONE
#
#
# Variables:
#
# SEQN
#     Respondent sequence number
#     -> whole-number identifier
#
#
# HOD050
#     Number of rooms in home
#     -> whole-number quantity / special code
#
#
# HOQ065
#     Home owned, bought, rented, other
#     -> categorical response code
#
#
# Therefore all 3 variables can safely become nullable Int64.
#
#
# IMPORTANT:
#
# This conclusion applies ONLY to HOQ_I.
#
# Do NOT generalize this to:
#
#     Dietary continuous variables
#     Laboratory continuous results
#     Examination measurements
#     survey/sample weights
#
# Those genuine decimal values must remain AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
#
# Attached HOQ_I contains NONE.
#
# Therefore:
#
#     DO NOT perform any tiny -> zero replacement.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 3 HOQ_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HOQ_I."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in HOQ_I."
    )


print(
    "PASS: HOQ_I contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# Unlike files such as ALQ_I and DIQ_I,
# this attachment has no XPORT zero corruption.
#
# Therefore source_df itself is the exact
# numerical fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero correction required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() below is used ONLY to detect fractional
# observations.
#
# It does NOT modify source data.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found in HOQ_I.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in HOQ_I."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 9. VALIDATE HOD050 AND HOQ065 ARE WHOLE-NUMBER
# ============================================================

for col in [
    "HOD050",
    "HOQ065"
]:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: HOD050 and HOQ065 contain only "
    "whole-number values/codes."
)


# ============================================================
# 10. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() during actual conversion.
#
# All fields have already been proven whole-number.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 3 HOQ_I variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. CDC CHECK - HOD050
# ============================================================

#
# HOD050:
#
#     Number of rooms in home.
#
# Count kitchen, but not bathroom.
#
#
# CDC notes:
#
# Responses >= 13 were grouped and released as:
#
#     13 = 13 or more
#
#
# CDC frequencies:
#
#     1   =   49
#     2   =  204
#     3   =  831
#     4   = 1810
#     5   = 2071
#     6   = 1728
#     7   = 1130
#     8   =  773
#     9   =  434
#     10  =  323
#     11  =  133
#     12  =   71
#     13  =   51
#     777 =   34
#     999 =    0
#
#     Missing = 329
#

EXPECTED_HOD050 = {
    1: 49,
    2: 204,
    3: 831,
    4: 1810,
    5: 2071,
    6: 1728,
    7: 1130,
    8: 773,
    9: 434,
    10: 323,
    11: 133,
    12: 71,
    13: 51,
    777: 34
}


actual_hod050 = (
    df["HOD050"]
    .value_counts()
    .sort_index()
    .to_dict()
)


hod050_missing = int(
    df["HOD050"]
    .isna()
    .sum()
)


hod050_nonmissing = int(
    df["HOD050"]
    .notna()
    .sum()
)


print(
    "\n--- HOD050 ---"
)

print(
    "Observed frequencies:"
)

print(
    actual_hod050
)

print(
    "Nonmissing:",
    f"{hod050_nonmissing:,}"
)

print(
    "Missing:",
    f"{hod050_missing:,}"
)


if actual_hod050 != EXPECTED_HOD050:

    raise ValueError(
        "HOD050 frequencies differ.\n\n"
        f"Expected:\n{EXPECTED_HOD050}\n\n"
        f"Actual:\n{actual_hod050}"
    )


if hod050_nonmissing != 9642:

    raise ValueError(
        "HOD050 nonmissing count differs."
    )


if hod050_missing != 329:

    raise ValueError(
        "HOD050 missing count differs."
    )


if int(
    df["HOD050"]
    .eq(999)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected HOD050=999 values."
    )


print(
    "PASS: HOD050 matches CDC exactly."
)


# ============================================================
# 12. HOD050 STANDARD ROOM RANGE CHECK
# ============================================================

#
# Standard numeric responses:
#
#     1 through 13
#
# 13 represents "13 or more".
#
# Refused:
#
#     777
#

standard_room_count = int(
    df["HOD050"]
    .between(
        1,
        13
    )
    .sum()
)


refused_room_count = int(
    df["HOD050"]
    .eq(777)
    .sum()
)


print(
    "\n--- HOD050 RANGE CHECK ---"
)

print(
    "1-13:",
    f"{standard_room_count:,}"
)

print(
    "777 Refused:",
    refused_room_count
)


if standard_room_count != 9608:

    raise ValueError(
        "HOD050 standard 1-13 count differs."
    )


if refused_room_count != 34:

    raise ValueError(
        "HOD050 refused count differs."
    )


if (
    standard_room_count
    +
    refused_room_count
) != 9642:

    raise ValueError(
        "HOD050 nonmissing total does not reconcile."
    )


print(
    "PASS: HOD050 range/special-code pattern validated."
)


# ============================================================
# 13. CDC CHECK - HOQ065
# ============================================================

#
# Home ownership:
#
#     1 = Owned or being bought
#         5,333
#
#     2 = Rented
#         4,121
#
#     3 = Other arrangement
#           162
#
#     7 = Refused
#            10
#
#     9 = Don't know
#            16
#
# Missing = 329
#

EXPECTED_HOQ065 = {
    1: 5333,
    2: 4121,
    3: 162,
    7: 10,
    9: 16
}


actual_hoq065 = (
    df["HOQ065"]
    .value_counts()
    .sort_index()
    .to_dict()
)


hoq065_nonmissing = int(
    df["HOQ065"]
    .notna()
    .sum()
)


hoq065_missing = int(
    df["HOQ065"]
    .isna()
    .sum()
)


print(
    "\n--- HOQ065 ---"
)

print(
    "Observed frequencies:"
)

print(
    actual_hoq065
)

print(
    "Nonmissing:",
    f"{hoq065_nonmissing:,}"
)

print(
    "Missing:",
    f"{hoq065_missing:,}"
)


if actual_hoq065 != EXPECTED_HOQ065:

    raise ValueError(
        "HOQ065 frequencies differ.\n\n"
        f"Expected:\n{EXPECTED_HOQ065}\n\n"
        f"Actual:\n{actual_hoq065}"
    )


if hoq065_nonmissing != 9642:

    raise ValueError(
        "HOQ065 nonmissing count differs."
    )


if hoq065_missing != 329:

    raise ValueError(
        "HOQ065 missing count differs."
    )


print(
    "PASS: HOQ065 matches CDC exactly."
)


# ============================================================
# 14. HOQ065 VALID RESPONSE CODE CHECK
# ============================================================

VALID_HOQ065_CODES = {
    1,
    2,
    3,
    7,
    9
}


observed_hoq065_codes = set(
    df["HOQ065"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)


unexpected_hoq065 = (
    observed_hoq065_codes
    -
    VALID_HOQ065_CODES
)


print(
    "\n--- HOQ065 VALID CODE CHECK ---"
)

print(
    "Observed codes:",
    sorted(
        observed_hoq065_codes
    )
)


if unexpected_hoq065:

    raise ValueError(
        "Unexpected HOQ065 codes found: "
        + str(
            sorted(
                unexpected_hoq065
            )
        )
    )


print(
    "PASS: No unexpected HOQ065 response codes."
)


# ============================================================
# 15. HOD050 / HOQ065 MISSING-MASK CHECK
# ============================================================

#
# Direct attachment:
#
# HOD050 missing = 329
# HOQ065 missing = 329
#
# Their missing positions match exactly.
#

hod_missing_mask = (
    df["HOD050"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


hoq_missing_mask = (
    df["HOQ065"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


missing_mask_mismatches = int(
    np.sum(
        hod_missing_mask
        !=
        hoq_missing_mask
    )
)


print(
    "\n--- HOUSING VARIABLE MISSING MASK ---"
)

print(
    "HOD050 missing:",
    int(
        hod_missing_mask.sum()
    )
)

print(
    "HOQ065 missing:",
    int(
        hoq_missing_mask.sum()
    )
)

print(
    "Missing-position mismatches:",
    missing_mask_mismatches
)


if missing_mask_mismatches != 0:

    raise ValueError(
        "HOD050 and HOQ065 missing positions differ."
    )


print(
    "PASS: HOD050 and HOQ065 missing "
    "positions match exactly."
)


# ============================================================
# 16. CONFIRM NO ZEROS CREATED
# ============================================================

#
# Source XPT:
#
#     tiny values    = 0
#     ordinary zeros = 0
#
# Therefore conversion must not create zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros were created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No numeric zeros created."
)


# ============================================================
# 17. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HOQ_I required NO numerical repair.
#
# Therefore every numeric value must remain identical
# to the source XPT.
#
# Formatting:
#
#     83732.0 -> 83732
#
# does not change its numerical value.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT numeric values exactly."
)


# ============================================================
# 18. EXPORT CSV
# ============================================================

#
# Standard fidelity export:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# HOQ_I itself contains no continuous decimals,
# but using the same export approach maintains
# consistency with files that do contain them.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 19. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 3 HOQ_I variables."
)


# ============================================================
# 21. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 22. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 23. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 24. RAW CSV .0 CHECK
# ============================================================

#
# HOQ_I contains no genuine continuous decimals.
#
# Therefore all 3 variables should contain no
# unnecessary ".0".
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HOQ_I variable."
)


# ============================================================
# 25. FINAL HOD050 FREQUENCY CHECK
# ============================================================

final_hod050 = pd.to_numeric(
    raw_csv_df["HOD050"]
    .where(
        raw_csv_df["HOD050"] != "",
        np.nan
    ),
    errors="coerce"
)


final_hod050_freq = (
    final_hod050
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


print(
    "\n--- FINAL HOD050 FREQUENCIES ---"
)

print(
    final_hod050_freq
)


if final_hod050_freq != EXPECTED_HOD050:

    raise ValueError(
        "Final HOD050 frequency distribution changed."
    )


print(
    "PASS: Final HOD050 frequencies preserved."
)


# ============================================================
# 26. FINAL HOQ065 FREQUENCY CHECK
# ============================================================

final_hoq065 = pd.to_numeric(
    raw_csv_df["HOQ065"]
    .where(
        raw_csv_df["HOQ065"] != "",
        np.nan
    ),
    errors="coerce"
)


final_hoq065_freq = (
    final_hoq065
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


print(
    "\n--- FINAL HOQ065 FREQUENCIES ---"
)

print(
    final_hoq065_freq
)


if final_hoq065_freq != EXPECTED_HOQ065:

    raise ValueError(
        "Final HOQ065 frequency distribution changed."
    )


print(
    "PASS: Final HOQ065 frequencies preserved."
)


# ============================================================
# 27. FINAL HOUSING MISSING-MASK CHECK
# ============================================================

final_hod_missing = (
    raw_csv_df["HOD050"]
    .eq("")
    .to_numpy(
        dtype=bool
    )
)


final_hoq_missing = (
    raw_csv_df["HOQ065"]
    .eq("")
    .to_numpy(
        dtype=bool
    )
)


final_mask_mismatch = int(
    np.sum(
        final_hod_missing
        !=
        final_hoq_missing
    )
)


print(
    "\n--- FINAL HOUSING MISSING MASK ---"
)

print(
    "HOD050 missing:",
    int(
        final_hod_missing.sum()
    )
)

print(
    "HOQ065 missing:",
    int(
        final_hoq_missing.sum()
    )
)

print(
    "Mask mismatches:",
    final_mask_mismatch
)


if int(
    final_hod_missing.sum()
) != 329:

    raise ValueError(
        "Final HOD050 missing count changed."
    )


if int(
    final_hoq_missing.sum()
) != 329:

    raise ValueError(
        "Final HOQ065 missing count changed."
    )


if final_mask_mismatch != 0:

    raise ValueError(
        "Final HOD050/HOQ065 missing mask changed."
    )


print(
    "PASS: Final housing missingness preserved."
)


# ============================================================
# 28. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected numeric zero values "
        "found in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 29. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 30. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "HOQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Housing Characteristics"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    INTEGER_COLUMNS
)


print(
    "HOD050 nonmissing:",
    f"{hod050_nonmissing:,}"
)


print(
    "HOD050 missing:",
    f"{hod050_missing:,}"
)


print(
    "HOD050 13-or-more category:",
    int(
        df["HOD050"]
        .eq(13)
        .sum()
    )
)


print(
    "HOQ065 owned/bought:",
    int(
        df["HOQ065"]
        .eq(1)
        .sum()
    )
)


print(
    "HOQ065 rented:",
    int(
        df["HOQ065"]
        .eq(2)
        .sum()
    )
)


print(
    "HOQ065 other arrangement:",
    int(
        df["HOQ065"]
        .eq(3)
        .sum()
    )
)


print(
    "Housing missing-mask mismatches:",
    missing_mask_mismatches
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "HOQ_I contains no genuine decimal variables."
)


print(
    "SEQN, HOD050 and HOQ065 can safely "
    "be exported as whole-number fields."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "variables remain exactly AS IS."
)

NHANES 2015-2016 HOQ_I XPT -> CSV
Rows: 9,971
Columns: 3
PASS: Structure = 9,971 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: ['SEQN', 'HOD050', 'HOQ065']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: HOQ_I contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero correction required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
HOD050: 0 fractional observations
HOQ065: 0 fractional observations

PASS: No genuine decimal variables found in HOQ_I.

--- SEQN ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN 83732-93702 validated.
PASS: HOD050 and HOQ065 contain only whole-number values/codes.

PASS: All 3 HOQ_I variables converted to nullable Int64 without rounding.

--- HOD050 ---
Observed frequencies:
{np.int64(1): 49, np.int64(2): 204, np.int64(3): 831, np.int64(4): 1810, np